# 립 채점표 체크별 예시 프롬프트: 비공개 Colab
LaPa 비상업 연구 전용. `rubric_examples.py export`로 만든 `rubric_examples_pack.zip`만 사용합니다. 체크마다 사람이 "예"·"아니오"라고 답한 예시 사진을 보여 주고, 검증용 16장을 문항별로 채점합니다(80회, T4 약 30~60분). 사람 점수는 묶음에 들어 있지 않습니다.

In [ ]:
!nvidia-smi

## 1. 비공개 Drive 마운트 및 데이터 무결성 확인
본인만 접근 가능한 `lip-rubric-private` 폴더에 `rubric_examples_pack.zip`을 두고 담당자가 알려 준 SHA-256을 입력하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, zipfile
DRIVE = Path('/content/drive/MyDrive/lip-rubric-private')
ZIP = DRIVE / 'rubric_examples_pack.zip'
EXPECTED_SHA256 = '여기에_담당자가_출력받은_SHA256_입력'
actual = hashlib.sha256(ZIP.read_bytes()).hexdigest()
assert actual.lower() == EXPECTED_SHA256.lower(), f'SHA-256 불일치: {actual}'
ROOT = Path('/content/lip-examples-pack'); ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for name in z.namelist():
        assert (ROOT / name).resolve().is_relative_to(ROOT.resolve()), '잘못된 압축 경로'
    z.extractall(ROOT)
print('검증 완료:', actual)

## 2. 의존성 설치
"세션 다시 시작" 안내가 떠도 무시하세요. 채점은 별도 프로세스로 실행됩니다.

In [ ]:
%pip install 'transformers==4.57.6' 'accelerate==1.12.0' 'bitsandbytes==0.49.0' 'qwen-vl-utils==0.0.14'
import subprocess, sys
(DRIVE / 'examples-pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))

## 3. 모델 버전 확인
judges.json의 VLM 리비전(2026-11-12 확인)을 그대로 씁니다. 실행 전에 모델 카드 라이선스를 다시 확인하세요.

In [ ]:
import json
cfg = json.loads((ROOT / 'judges.json').read_text())
print(cfg['vlm']['model_id'], cfg['vlm']['revision'], cfg['vlm']['license'])
assert cfg['vlm']['revision'], 'judges.json에 VLM 리비전이 필요합니다'

## 4. 체크별 예시 프롬프트 실행
사진마다 문항 5개를 따로 묻습니다. 결과는 Drive의 `rubric_examples_results.csv`에 한 줄씩 바로 저장되고, 끊기면 1단계부터 다시 실행하면 이어서 진행합니다.

In [ ]:
result = DRIVE / 'rubric_examples_results.csv'
subprocess.run([sys.executable, str(ROOT/'rubric_examples.py'), 'run', '--pack', str(ROOT), '--output', str(result)], cwd=ROOT, check=True)
print('저장 위치:', result)

## 5. 결과 내려받기
`rubric_examples_results.csv`를 담당자의 `research/data/lip_rubric/results/`에 저장합니다. 실험이 끝나면 Drive의 비공개 폴더와 Colab 런타임을 삭제합니다.